# 🖥️ Colab Pro Survival Guide — GPU Budget, Session Management & Model Caching

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/00_colab_pro_survival_guide.ipynb)

> **Read this first.** This notebook is the operational manual for running the
> Castalia AI-engineering course (~190 notebooks, 8 domains) on Google Colab Pro.
> It covers GPU budgeting, session management, model caching, cost optimization,
> and troubleshooting — everything you need to avoid wasting compute units
> and lost work.

## Why You Need This Guide

Colab Pro gives you powerful GPUs on demand, but it is **not unlimited**.
Every minute of GPU time costs compute units, every disconnection risks lost
work, and every redundant model download wastes 10–15 minutes you could spend
learning.

Students who read this guide before starting the course typically:
- **Save 30–50% on compute units** by using CPU runtimes where GPUs aren't needed
- **Avoid the #1 frustration** — "CUDA out of memory" errors — by understanding VRAM budgets
- **Never lose work** by setting up Drive-based checkpointing from day one
- **Cut model loading time in half** by caching weights on Google Drive

---
## 1 💳 Understanding Colab Pro Tiers

Google offers three Colab tiers. The table below captures the differences
that matter for this course.

| Feature | Free | Pro ($11.49/mo) | Pro+ ($58.49/mo) |
|---|---|---|---|
| **Compute units / month** | Limited burst | 100 | 500 |
| **GPU priority** | Low (often queued) | Standard | High |
| **Available GPUs** | T4 (sometimes) | T4, L4 | T4, L4, A100 (40 GB) |
| **System RAM** | ~12 GB | ~25–50 GB (high-RAM option) | ~50–80 GB |
| **Idle timeout** | ~30 min | ~90 min | ~90 min |
| **Max session length** | ~12 h | ~24 h | ~24 h |
| **Background execution** | ❌ | ❌ | ✅ (sessions run with lid closed) |
| **Terminal access** | ❌ | ✅ | ✅ |

> **Recommendation for this course:** Colab **Pro** is the sweet spot.
> Most notebooks run on a T4 with 4-bit quantization, and 100 units/month
> is enough if you follow the cost-optimization tips below. Pro+ is only
> necessary if you want to run 70 B+ models or need background execution.

### How Compute Units Work

Compute units (CU) are consumed whenever a **GPU or TPU runtime** is active.
CPU-only runtimes are free (or nearly free). The burn rate depends on the
hardware:

| GPU | Approx. CU / hour |
|---|---|
| T4 | ~1.5 |
| L4 | ~3.2 |
| A100 (40 GB) | ~8.0 |

So a two-hour session on a T4 costs roughly **3 compute units**. Keep this
in mind when choosing hardware — an A100 burns units 5× faster than a T4.

---
## 2 🎮 GPU Types and When You Get What

| GPU | VRAM | Architecture | Availability |
|---|---|---|---|
| **T4** | 16 GB | Turing (2018) | Most common; default for Pro |
| **L4** | 24 GB | Ada Lovelace (2023) | Newer; sometimes offered as upgrade |
| **A100** | 40 GB | Ampere (2020) | Pro+ only; high priority queue |

The **T4** is what you will get 80–90% of the time on Colab Pro, and that is
perfectly fine. The Castalia course is designed around T4-class hardware
with 4-bit quantized models.

**Rule of thumb:** if a notebook title mentions a 7 B model, T4 is enough.
If it mentions 70 B, you need an A100 or should use an API-based approach.

### Choosing a GPU in Colab

`Runtime → Change runtime type → Hardware accelerator → GPU → T4 / L4 / A100`

If the GPU you want is not available, try:
1. Wait 10–15 minutes and reconnect
2. Try a different time of day (US evenings = peak)
3. Accept a smaller GPU and use more aggressive quantization

---
## 3 📊 VRAM Budgeting

The single most important equation for running LLMs on Colab:

```
VRAM (GB) ≈ parameters × bytes_per_parameter
```

**Bytes per parameter by precision:**
- `fp32` → 4 bytes
- `fp16 / bf16` → 2 bytes
- `int8` → 1 byte
- `4-bit (GPTQ / AWQ / GGUF Q4)` → 0.5 bytes

This is the **model weight memory** only. You also need headroom for:
- KV cache (grows with sequence length)
- Activations during inference
- CUDA kernels and framework overhead (~1–2 GB)

A practical rule: keep **2–3 GB free** after loading weights.

### VRAM Budget Table

| Model | Precision | Weight Memory | Fits on T4 (16 GB)? | Fits on A100 (40 GB)? |
|---|---|---|---|---|
| 1.5 B (Qwen-2.5, SmolLM) | fp16 | ~3 GB | ✅ Easily | ✅ |
| 7 B (Llama-3.1, Mistral) | fp16 | ~14 GB | ⚠️ Tight (no room for KV) | ✅ |
| **7 B** | **4-bit** | **~3.5 GB** | **✅ Easily** ← course default | ✅ |
| 14 B (Qwen-2.5) | 4-bit | ~7 GB | ✅ Comfortable | ✅ |
| 34 B (CodeLlama) | 4-bit | ~17 GB | ❌ | ✅ |
| 70 B (Llama-3.1) | 4-bit | ~35 GB | ❌ | ⚠️ Tight |

> **The course primarily uses 7 B models at 4-bit quantization (~3.5 GB).**
> This leaves ~12 GB free on a T4 for KV cache, batching, and experimentation.

In [ ]:
# ── Check your GPU and calculate available VRAM budget ──
import subprocess, re

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

# Parse total and used memory
match = re.search(r"(\d+)MiB\s*/\s*(\d+)MiB", result.stdout)
if match:
    used_mb, total_mb = int(match.group(1)), int(match.group(2))
    free_gb = (total_mb - used_mb) / 1024
    print(f"{'='*50}")
    print(f"Total VRAM : {total_mb / 1024:.1f} GB")
    print(f"Used VRAM  : {used_mb / 1024:.1f} GB")
    print(f"Free VRAM  : {free_gb:.1f} GB")
    print(f"\nCan fit a 7B model at 4-bit?  {'YES ✅' if free_gb > 5 else 'NO ❌'}")
    print(f"Can fit a 14B model at 4-bit? {'YES ✅' if free_gb > 9 else 'NO ❌'}")
    print(f"Can fit a 70B model at 4-bit? {'YES ✅' if free_gb > 37 else 'NO ❌'}")
else:
    print("⚠️  No GPU detected. Switch to a GPU runtime:")
    print("   Runtime → Change runtime type → GPU")

---
## 4 ⏱️ Session Management

Understanding Colab session lifecycle prevents lost work and wasted compute.

### Idle Timeout

| Tier | Idle Timeout | Max Session Length |
|---|---|---|
| Free | ~30 minutes | ~12 hours |
| Pro | ~90 minutes | ~24 hours |
| Pro+ | ~90 minutes | ~24 hours |

"Idle" means **no code execution and no browser interaction**. If you walk
away to make coffee, your 30-minute free-tier session may already be dead
when you return.

**Tips to stay connected:**
- Keep the Colab tab **visible** (not minimized) — Colab sends keep-alive
  pings when the tab is focused
- Run a lightweight cell periodically (even `print("alive")` resets the timer)
- Do not rely on the browser auto-reconnect; it often fails silently

### What Happens When You Disconnect

1. **All Python variables are lost** — your model, tokenizer, dataframes, everything
2. **Files in `/content/`** survive until the runtime VM is recycled (could be minutes or hours)
3. **Files on Google Drive** (if mounted) are safe and persistent
4. **Installed pip packages are lost** — you must re-run `!pip install ...` cells
5. **Environment variables are lost** — including `HF_HOME`, `HF_TOKEN`, etc.

> 💡 **Lesson:** Always save important outputs (trained adapters, eval results,
> generated text) to **Google Drive**, not just `/content/`.

In [ ]:
# ── Auto-checkpoint to Google Drive ──
# Run this cell at the start of any long notebook to set up saving.

import os, shutil
from pathlib import Path

def mount_drive():
    """Mount Google Drive if not already mounted."""
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    return Path("/content/drive/MyDrive")

def save_checkpoint(src_path: str, drive_folder: str = "castalia_checkpoints"):
    """
    Copy a file or directory from /content/ to Google Drive.
    Call this after training steps, eval runs, or any work you want to keep.
    """
    drive_root = mount_drive()
    dest = drive_root / drive_folder
    dest.mkdir(parents=True, exist_ok=True)

    src = Path(src_path)
    if src.is_dir():
        shutil.copytree(src, dest / src.name, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dest / src.name)
    print(f"✅ Saved {src.name} → {dest / src.name}")

# Example usage:
# save_checkpoint("/content/my_lora_adapter")
# save_checkpoint("/content/eval_results.json")
print("Checkpoint helper loaded.  Use: save_checkpoint(path)")

---
## 5 📦 Model Caching Strategy

By default, HuggingFace downloads models to `~/.cache/huggingface/` which
lives on the **ephemeral Colab VM disk**. Every time your runtime recycles,
those 4–8 GB downloads happen again. For a 190-notebook course, that is
potentially **hundreds of gigabytes** of redundant downloads and **hours** of
wasted time.

The fix: redirect the HuggingFace cache to **Google Drive**.

### How It Works

1. Mount Google Drive
2. Set `HF_HOME` environment variable to a Drive path
3. All `transformers`, `datasets`, and `huggingface_hub` downloads go to Drive
4. Next session: same models load from Drive in seconds instead of minutes

**Time savings per model:**

| Model Size | Download Time (first) | Load from Drive (cached) |
|---|---|---|
| 1.5 B (4-bit) | ~1–2 min | ~15 sec |
| 7 B (4-bit) | ~5–8 min | ~30 sec |
| 14 B (4-bit) | ~10–15 min | ~1 min |

Over 190 notebooks, caching saves **10–20+ hours** of download time.

In [ ]:
# ════════════════════════════════════════════════════════════
#  STANDARD COLAB SETUP CELL — copy this into every notebook
# ════════════════════════════════════════════════════════════
import os

# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# 2. Redirect HuggingFace cache to Drive (avoids re-downloading models)
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

# 3. (Optional) Set your HuggingFace token for gated models like Llama
# os.environ["HF_TOKEN"] = "hf_..."  # or use Colab Secrets (recommended)

# 4. Verify
cache_path = os.environ["HF_HOME"]
os.makedirs(cache_path, exist_ok=True)
print(f"✅ HF cache → {cache_path}")
print(f"   Drive space used by cache: ", end="")
!du -sh {cache_path} 2>/dev/null || echo "(empty — first run)"

### ⚠️ Managing Drive Storage

Google Drive gives you **15 GB free** (or 100 GB+ with Google One). Model
caches can grow quickly:

- A single 7 B 4-bit model: ~4 GB
- The full course cache (all unique models): ~20–30 GB

**If you are running low on Drive space:**

```python
# List cached models by size
!du -h --max-depth=2 /content/drive/MyDrive/hf_cache/hub/ | sort -rh | head -20

# Remove a specific model
!rm -rf /content/drive/MyDrive/hf_cache/hub/models--old-model-name
```

---
## 6 💰 Compute Unit Budget for the Course

Let’s estimate how many compute units you need to complete the full course.

### Estimation

| Factor | Estimate |
|---|---|
| Total notebooks | ~190 |
| Notebooks needing GPU | ~130 (the rest are text / prompt engineering) |
| Average GPU time per notebook | ~45 min |
| Average CU per notebook (T4) | ~1.1 |
| **Total GPU notebooks CU** | **~145 CU** |

With Colab Pro (100 CU/month), plan for **roughly 2 months** of GPU time.
If you are efficient with CPU runtimes and session management, you can
potentially finish in 6–8 weeks.

### Which Domains Need GPU vs CPU?

| Domain | Notebooks | GPU Needed? | Notes |
|---|---|---|---|
| **Prompt Engineering** | ~25 | 🟢 Mostly CPU | API-based; use CPU runtime |
| **RAG** | ~25 | 🟡 Some GPU | Embedding models fit on CPU; rerankers need GPU |
| **Fine-Tuning** | ~20 | 🔴 Always GPU | Training requires CUDA |
| **Evaluation** | ~25 | 🟡 Some GPU | LLM-as-judge needs GPU; metric-based evals are CPU |
| **Agents** | ~25 | 🟡 Some GPU | Tool-use notebooks may use local models |
| **Systems** | ~22 | 🔴 Always GPU | Inference runtimes, benchmarking |
| **ML Fundamentals** | ~25 | 🟡 Mixed | Theory notebooks are CPU; training demos need GPU |
| **Multimodal** | ~20 | 🔴 Always GPU | Vision and audio models need VRAM |

> 💡 **Quick win:** Switch to **CPU runtime** for prompt engineering and
> evaluation notebooks. This alone can save **20–30 compute units**.

In [ ]:
# ── Compute unit estimator ──
# Adjust these numbers based on your pace

total_notebooks = 190
gpu_notebooks = 130
cpu_notebooks = total_notebooks - gpu_notebooks

avg_gpu_minutes = 45          # average GPU time per notebook
cu_per_hour_t4 = 1.5          # T4 burn rate

total_cu = gpu_notebooks * (avg_gpu_minutes / 60) * cu_per_hour_t4
pro_months = total_cu / 100   # Pro gives 100 CU/month

print(f"Estimated compute units needed: {total_cu:.0f} CU")
print(f"Colab Pro months needed:        {pro_months:.1f} months")
print(f"CPU-only notebooks (free):      {cpu_notebooks}")
print(f"\n💡 Tip: if you batch GPU notebooks efficiently,")
print(f"   you can reduce this by ~20% (startup overhead savings).")

---
## 7 💸 Cost Optimization Tactics

Every compute unit counts. Here are battle-tested strategies to get the
most out of your Colab Pro subscription.

### Tactic 1: Use T4 Unless You Specifically Need More

The T4 is the **cheapest GPU** on Colab (~1.5 CU/hr vs ~8 CU/hr for A100).
Since the course is designed for T4 + 4-bit quantization, you should rarely
need to upgrade. The A100 costs **5× more per hour** — only use it when a
notebook explicitly requires it.

### Tactic 2: Batch GPU-Heavy Notebooks

Each GPU session has startup overhead (~2–3 minutes for runtime allocation,
package installation, model download). If you have three GPU notebooks to
complete, doing them in **one session** saves ~6–9 minutes of overhead.

Plan your study sessions: "Today I will do Systems notebooks 5–8" rather
than switching between GPU and CPU notebooks randomly.

### Tactic 3: Clean Up VRAM Between Experiments

Instead of restarting the runtime (which forces re-installation of packages),
use targeted memory cleanup between model loads.

In [ ]:
# ── VRAM cleanup utility ──
import gc
import torch

def clear_vram(*objects_to_delete):
    """
    Free GPU memory between experiments.
    Pass any model/tensor objects you want to explicitly delete.

    Usage:
        clear_vram(model, tokenizer)
        # Now load a different model
    """
    for obj in objects_to_delete:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_mem / 1e9
        print(f"🧹 VRAM after cleanup: {used:.1f} / {total:.1f} GB used")
    else:
        print("No GPU available.")

# Example: clear_vram(model, tokenizer)
print("clear_vram() loaded.  Use: clear_vram(model, tokenizer)")

### Tactic 4: Use CPU Runtime for Non-GPU Notebooks

This is the **single biggest cost saver**. Change your runtime to CPU
(`Runtime → Change runtime type → None`) for:

- Prompt engineering notebooks (API calls only)
- Text-based evaluation (string metrics, human eval setup)
- Data preprocessing and exploration
- Any notebook that does not import `torch` or run a local model

CPU runtimes consume **zero compute units**.

### Tactic 5: Disconnect When Done

Do not leave GPU runtimes idling. When you finish a notebook:

1. Save any outputs to Drive
2. `Runtime → Disconnect and delete runtime`

This immediately stops compute unit consumption. The 90-minute idle timeout
is a safety net, not a strategy.

In [ ]:
# ── Memory monitoring utility ──
# Run this cell to see current resource usage at a glance

import psutil

def system_status():
    """Print a snapshot of system resources."""
    # RAM
    ram = psutil.virtual_memory()
    print(f"{'='*50}")
    print(f"  SYSTEM RESOURCE STATUS")
    print(f"{'='*50}")
    print(f"RAM:  {ram.used / 1e9:.1f} / {ram.total / 1e9:.1f} GB "
          f"({ram.percent}% used)")

    # Disk
    disk = psutil.disk_usage("/")
    print(f"Disk: {disk.used / 1e9:.1f} / {disk.total / 1e9:.1f} GB "
          f"({disk.percent}% used)")

    # GPU
    try:
        import torch
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                props = torch.cuda.get_device_properties(i)
                used = torch.cuda.memory_allocated(i) / 1e9
                total = props.total_mem / 1e9
                print(f"GPU {i} ({props.name}): "
                      f"{used:.1f} / {total:.1f} GB VRAM")
        else:
            print("GPU:  ❌ No GPU (CPU runtime)")
    except ImportError:
        print("GPU:  torch not installed")
    print()

system_status()

---
## 8 📁 Google Drive Integration

Google Drive is your **persistent storage layer** on Colab. Everything on
the Colab VM is temporary; everything on Drive survives across sessions.

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount("/content/drive")

# Verify
!ls /content/drive/MyDrive/ | head -10
print("\n✅ Drive mounted at /content/drive/MyDrive/")

### Recommended Folder Structure

Create this structure on your Google Drive for the course:

```
MyDrive/
├── hf_cache/                   # HuggingFace model & dataset cache
│   └── hub/                    # Model weights live here
├── castalia_checkpoints/       # Saved training checkpoints
│   ├── fine_tuning/
│   ├── rag_indexes/
│   └── eval_results/
├── castalia_outputs/           # Generated outputs, logs, exports
│   ├── benchmark_results/
│   ├── generated_text/
│   └── figures/
└── castalia_data/              # Custom datasets, documents for RAG
```

In [ ]:
# ── Create the recommended folder structure ──
import os

folders = [
    "/content/drive/MyDrive/hf_cache",
    "/content/drive/MyDrive/castalia_checkpoints/fine_tuning",
    "/content/drive/MyDrive/castalia_checkpoints/rag_indexes",
    "/content/drive/MyDrive/castalia_checkpoints/eval_results",
    "/content/drive/MyDrive/castalia_outputs/benchmark_results",
    "/content/drive/MyDrive/castalia_outputs/generated_text",
    "/content/drive/MyDrive/castalia_outputs/figures",
    "/content/drive/MyDrive/castalia_data",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"📁 {folder.replace('/content/drive/MyDrive/', '')}")

print("\n✅ Folder structure created on Google Drive.")

---
## 9 🛠️ Troubleshooting Common Colab Issues

These are the issues you **will** encounter. Bookmark this section.

### ❌ "CUDA out of memory"

The most common error. Your model + KV cache + overhead exceeds VRAM.

**Fixes (in order):**
1. **Reduce batch size** — try `batch_size=1`
2. **Use 4-bit quantization** — `load_in_4bit=True` with `BitsAndBytesConfig`
3. **Reduce `max_new_tokens`** — smaller outputs = smaller KV cache
4. **Restart runtime** — `Runtime → Restart runtime` (clears leaked memory)
5. **Clear VRAM** — use `clear_vram()` from Tactic 3 above
6. **Use a smaller model** — 7 B instead of 14 B, or 1.5 B for quick experiments

```python
# Quick fix: load any model in 4-bit
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config
)
```

### 🔌 "Runtime disconnected" / "Session crashed"

**What happened:** Colab killed your session (idle timeout, resource limit, or OOM crash).

**Recovery steps:**
1. Click "Reconnect" (or `Runtime → Reconnect`)
2. If that fails, `Runtime → Disconnect and delete runtime`, then reconnect
3. Re-mount Google Drive: `drive.mount("/content/drive")`
4. Re-set environment variables (`HF_HOME`, `HF_TOKEN`)
5. Re-run your setup / import cells

> If you used Drive-based caching, you will **not** need to re-download models.

### 🚫 "Cannot connect to GPU" / GPU unavailable

Colab GPUs are a **shared resource**. During peak hours, you may be queued
or denied.

**Strategies:**
- Try different times: early morning (US) or late night tend to have better availability
- Try a different GPU type: if A100 is unavailable, T4 usually works
- Use CPU runtime for non-GPU notebooks while waiting
- If stuck for >30 min, consider using the notebook in CPU mode with API calls
  instead of local model inference

### 📦 Package version conflicts

Colab pre-installs many packages, sometimes at versions that conflict with
what a notebook needs.

**Fixes:**

```python
# Force reinstall a specific version
!pip install --force-reinstall transformers==4.45.0

# If you get "cannot import" after pip install, restart runtime
# Runtime → Restart runtime (then re-run import cells)

# Nuclear option: uninstall and reinstall
!pip uninstall -y transformers && pip install transformers
```

Many course notebooks pin exact versions in their first cell. **Always run
the install cell first**, even if you think you have the package.

In [ ]:
# ── Diagnostic snapshot ──
# Run this cell when something is not working.
# Share the output if asking for help.

import sys, importlib, subprocess

print("=" * 60)
print("  COLAB DIAGNOSTIC SNAPSHOT")
print("=" * 60)

# Python version
print(f"\nPython: {sys.version}")

# GPU info
gpu_result = subprocess.run(
    ["nvidia-smi",
     "--query-gpu=name,memory.total,memory.used,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu_result.returncode == 0:
    print(f"GPU:    {gpu_result.stdout.strip()}")
else:
    print("GPU:    ❌ Not available")

# Key packages
packages = [
    "torch", "transformers", "bitsandbytes", "peft",
    "datasets", "accelerate", "vllm", "trl", "sentencepiece",
]
print("\nInstalled packages:")
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "(no version)")
        print(f"  ✅ {pkg:20s} {version}")
    except ImportError:
        print(f"  ❌ {pkg:20s} NOT INSTALLED")

# CUDA
try:
    import torch
    print(f"\nCUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version:   {torch.version.cuda}")
        print(f"cuDNN version:  {torch.backends.cudnn.version()}")
except Exception as e:
    print(f"\nCUDA check failed: {e}")

# HF cache
import os
hf_home = os.environ.get("HF_HOME", "~/.cache/huggingface")
print(f"\nHF_HOME: {hf_home}")
print("=" * 60)

---
## 10 🎯 Key Takeaways

| # | Takeaway |
|---|---|
| 1 | **Cache models to Google Drive** — set `HF_HOME` in every notebook |
| 2 | **Use CPU runtimes** for prompt engineering and text-based evals |
| 3 | **T4 + 4-bit quantization** handles 90% of course notebooks |
| 4 | **Save outputs to Drive**, never rely on `/content/` persistence |
| 5 | **Disconnect idle runtimes** — don’t burn CU while reading docs |
| 6 | **Batch GPU work** — do multiple GPU notebooks per session |
| 7 | **Run diagnostics first** when troubleshooting (Section 9) |
| 8 | **Plan for ~2 months** of Pro to complete all GPU notebooks |

### 🗒️ Quick Reference: Standard Setup Cell

Copy this into the first cell of every course notebook:

```python
# ── Castalia Colab Setup ──
import os
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
```

That’s it. Three lines that save hours over the course of 190 notebooks.

---
## 📝 Exercises

Complete these before moving to the first real course notebook.

### Exercise 1: Verify Your GPU

Run the GPU check cell from Section 3. Answer:

1. What GPU did Colab assign you?
2. How much free VRAM do you have?
3. What is the largest 4-bit quantized model you could load?

### Exercise 2: Set Up Drive Caching

1. Run the caching setup cell from Section 5
2. Run the folder structure creation cell from Section 8
3. Verify: open Google Drive in another tab and confirm the folders exist
4. Download a small model to test the cache:

```python
from huggingface_hub import snapshot_download
snapshot_download("HuggingFaceTB/SmolLM2-135M")
```

5. Disconnect and reconnect the runtime, then re-run the download.
   It should be nearly instant (loaded from Drive cache).

### Exercise 3: Practice VRAM Budgeting

Without running any code, estimate the VRAM required for each scenario.
Then check your answers against the budget table in Section 3.

1. Loading Llama-3.1-8B at fp16
2. Loading Llama-3.1-8B at 4-bit quantization
3. Loading Qwen-2.5-14B at 4-bit quantization
4. Which of the above fit on a T4 (16 GB VRAM)?

---

> **You are ready.** Open the first domain notebook and start learning.
> Return to this guide whenever you hit a Colab-related issue — the
> troubleshooting section has your back. 🚀